# Notebook 06 — Real Access Log: Regex vs RF Detection

Two independent methods on an **unlabeled real access log**:

1. **Regex scanner** — 15 SQLi patterns applied to decoded full URL
2. **RF model (Way 3)** — NB03 Random Forest on extracted query values + 3-tier policy

| Agreement | Meaning |
|---|---|
| Both flag | High-confidence — two independent methods agree |
| RF only | Obfuscated/encoded payload regex missed |
| Regex only | Below RF threshold — weak signal or FP |
| Neither | Benign |

Ground-truth metrics computed automatically if log uses the
`line_no,label,...` format from the labeled dataset.


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import joblib, re, os, json, urllib.parse, time, warnings
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack, csr_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.makedirs('results/nb06', exist_ok=True)

SYMBOLS = ["'", '"', ";", "--", "#", "/*", "*/", "*", "+", "|",
           "(", ")", ">", "<", "\\", "/", "="]

def extract_symbol_frequencies(text):
    c = Counter()
    for sym in SYMBOLS:
        c[sym] = str(text).count(sym)
    return c

def build_symbol_matrix(queries):
    rows = [extract_symbol_frequencies(q) for q in queries]
    return csr_matrix(pd.DataFrame(rows).fillna(0).values)

def extract_query_values(url):
    parsed = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
    values = [urllib.parse.unquote(v).strip()
              for vlist in params.values() for v in vlist
              if urllib.parse.unquote(v).strip()]
    return ' '.join(values) if values else None

METHOD_RE = re.compile(
    r'"(GET|POST|HEAD|PUT|DELETE|OPTIONS|PATCH|TRACE|CONNECT)\s+([^"]+)\s+HTTP')

print('Setup complete.')


Setup complete.


## 2. SQLi Regex Scanner (15 Patterns)

In [2]:
SQLI_PATTERNS = {
    'UNION SELECT'     : r'union\s+(all\s+)?select',
    'Boolean OR/AND'   : r"'\s*(or|and)\s+[\w'\"(]",
    'Numeric tautology': r'(or|and)\s+\d+\s*=\s*\d+',
    'String tautology' : r"'\s*=\s*'",
    'SQL kw + clause'  : r'\b(select|insert|update|delete|drop|create|alter|exec|execute|cast|convert)\b.{0,80}\b(from|into|table|where|set)\b',
    'Time-based blind' : r'\b(sleep|benchmark|pg_sleep|waitfor\s+delay)\s*\(',
    'Subquery SELECT'  : r'\bselect\b.+\bfrom\b',
    'Hex encoding'     : r'0x[0-9a-fA-F]{4,}',
    'CHAR construct'   : r'char\s*\(\s*\d+',
    'CONCAT'           : r'concat\s*\(',
    'Info functions'   : r'\b(version|user|database|schema)\s*\(\s*\)',
    'System tables'    : r'\b(information_schema|sysobjects|syscolumns|pg_tables)\b',
    'Comment+payload'  : r"(.+)('|\"|;)\s*(--|#|/\*)",
    'Stacked queries'  : r';\s*(select|insert|update|delete|drop|exec)\b',
    'OR quote tautol.' : r"or\s+'[^']*'\s*=\s*'[^']*'",
}

COMPILED = {name: re.compile(pat, re.IGNORECASE)
            for name, pat in SQLI_PATTERNS.items()}

def regex_scan(url_decoded):
    return [name for name, pat in COMPILED.items() if pat.search(url_decoded)]

# Sanity check
ATTACK_TESTS = [
    "1' UNION SELECT username,password FROM users--",
    "(SELECT 7505 FROM(SELECT COUNT(*),CONCAT(0x7171787671,(SELECT version()))a)b)",
    "1'/**/UNION/**/SELECT/**/username,password/**/FROM/**/users--",
    "1' AND (SELECT 1 FROM dual WHERE SLEEP(3))--",
    "-1055' union all select 7758,7758,7758--",
    "admin'--",
    "1; DROP TABLE users--",
    "' OR ''='",
]
BENIGN_TESTS = ['106', 'laptop', 'cairo', 'price asc', '19519415', 'page 3']

print('=== REGEX SANITY CHECK ===')
all_ok = True
for t in ATTACK_TESTS:
    hits = regex_scan(t)
    sym = '✅' if hits else '❌'
    if not hits: all_ok = False
    print(f'  {sym} ATTACK  {repr(t[:60])}')
for t in BENIGN_TESTS:
    hits = regex_scan(t)
    sym = '✅' if not hits else '❌ FP'
    if hits: all_ok = False
    print(f'  {sym} BENIGN  {repr(t)}')
print()
print(f'All patterns OK: {all_ok}')
print(f'Patterns loaded: {len(SQLI_PATTERNS)}')
for name, pat in SQLI_PATTERNS.items():
    print(f'  {name:22s}  {pat[:55]}')


=== REGEX SANITY CHECK ===
  ✅ ATTACK  "1' UNION SELECT username,password FROM users--"
  ✅ ATTACK  '(SELECT 7505 FROM(SELECT COUNT(*),CONCAT(0x7171787671,(SELEC'
  ✅ ATTACK  "1'/**/UNION/**/SELECT/**/username,password/**/FROM/**/users-"
  ✅ ATTACK  "1' AND (SELECT 1 FROM dual WHERE SLEEP(3))--"
  ✅ ATTACK  "-1055' union all select 7758,7758,7758--"
  ✅ ATTACK  "admin'--"
  ✅ ATTACK  '1; DROP TABLE users--'
  ✅ ATTACK  "' OR ''='"
  ✅ BENIGN  '106'
  ✅ BENIGN  'laptop'
  ✅ BENIGN  'cairo'
  ✅ BENIGN  'price asc'
  ✅ BENIGN  '19519415'
  ✅ BENIGN  'page 3'

All patterns OK: True
Patterns loaded: 15
  UNION SELECT            union\s+(all\s+)?select
  Boolean OR/AND          '\s*(or|and)\s+[\w'\"(]
  Numeric tautology       (or|and)\s+\d+\s*=\s*\d+
  String tautology        '\s*=\s*'
  SQL kw + clause         \b(select|insert|update|delete|drop|create|alter|exec|e
  Time-based blind        \b(sleep|benchmark|pg_sleep|waitfor\s+delay)\s*\(
  Subquery SELECT         \bselect\b.+\bfrom\b
  H

## 3. Load Real Access Log

In [3]:
LOG_PATH = '../logs/access.log'   # ← change to your real log path

records = []
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for raw_line in f:
        line = raw_line.strip()
        if not line:
            continue
        # Labeled format: "line_no,label,<rest>"
        if re.match(r'^\d+,[01],', line):
            parts = line.split(',', 2)
            if len(parts) == 3:
                records.append({'log_entry': parts[2],
                                'has_label': True,
                                'true_label': int(parts[1])})
        else:
            records.append({'log_entry': line,
                            'has_label': False,
                            'true_label': -1})

df = pd.DataFrame(records)
HAS_LABELS = bool(df['has_label'].any())

total = len(df)
print(f'Log loaded  : {total:,} entries')
print(f'Has labels  : {HAS_LABELS}')
if HAS_LABELS:
    n_atk = int((df['true_label'] == 1).sum())
    print(f'True attacks: {n_atk}  ({n_atk / total * 100:.4f}%)')
    print(f'True benign : {total - n_atk:,}')


Log loaded  : 929,566 entries
Has labels  : False


## 4. Parse & Decode URLs

In [4]:
def parse_url(log_entry):
    m = METHOD_RE.search(str(log_entry))
    if not m:
        return ''
    url = urllib.parse.unquote(m.group(2))
    url = re.sub(r'utm_[a-z]+=[^&]*', '', url, flags=re.IGNORECASE).strip()
    return url

t0 = time.perf_counter()
df['url_decoded']  = df['log_entry'].apply(parse_url)
df['query_values'] = df['url_decoded'].apply(extract_query_values)
parse_ms = (time.perf_counter() - t0) * 1000

before = len(df)
df = df[df['url_decoded'] != ''].reset_index(drop=True)
after  = len(df)

no_qs   = int(df['query_values'].isna().sum())
with_qs = int(df['query_values'].notna().sum())

print(f'Parse time        : {parse_ms:.0f}ms')
print(f'Filtered (no parse): {before - after:,} removed (failed METHOD_RE match)')
print(f'Working set       : {len(df):,} entries')
print(f'With query string : {with_qs:,}  ({with_qs / len(df) * 100:.1f}%)')
print(f'Path-only (no QS) : {no_qs:,}  ({no_qs / len(df) * 100:.1f}%)')
print()
print('Sample decoded URLs:')
for url in df['url_decoded'].head(5):
    print(f'  {url[:90]}')


Parse time        : 7406ms
Filtered (no parse): 31 removed (failed METHOD_RE match)
Working set       : 929,535 entries
With query string : 8,988  (1.0%)
Path-only (no QS) : 920,547  (99.0%)

Sample decoded URLs:
  /
  /favicon.ico
  /www/
  /www/sql_backups/
  /www/sql_backups/


## 5. Regex Scan — Full Log

In [5]:
t0 = time.perf_counter()
df['regex_hits'] = df['url_decoded'].apply(regex_scan)
df['regex_flag'] = df['regex_hits'].apply(lambda x: 1 if x else 0)
regex_ms = (time.perf_counter() - t0) * 1000

n_regex = int(df['regex_flag'].sum())
regex_flagged = df[df['regex_flag'] == 1]

print(f'Regex scan time : {regex_ms:.0f}ms')
print(f'Regex flagged   : {n_regex:,}  ({n_regex / len(df) * 100:.4f}% of log)')
print()

pattern_counts = Counter(hit for hits in df['regex_hits'] for hit in hits)
print('Pattern breakdown (most common first):')
for name, count in pattern_counts.most_common():
    print(f'  {name:22s} : {count:,}')

print()
print(f'Flagged entries — first 20:')
for _, row in regex_flagged.head(20).iterrows():
    lbl = f'  [true_label={row["true_label"]}]' if row['has_label'] else ''
    print(f'  {row["url_decoded"][:85]}{lbl}')
    print(f'    → {row["regex_hits"]}')


Regex scan time : 13783ms
Regex flagged   : 199  (0.0214% of log)

Pattern breakdown (most common first):
  SQL kw + clause        : 67
  Subquery SELECT        : 65
  Hex encoding           : 46
  String tautology       : 45
  CONCAT                 : 33
  Time-based blind       : 32
  Boolean OR/AND         : 27
  UNION SELECT           : 27
  Numeric tautology      : 23
  Stacked queries        : 18
  System tables          : 13
  Comment+payload        : 12
  CHAR construct         : 11
  Info functions         : 5

Flagged entries — first 20:
  /test?x=88) AND 9535=6081 AND (6048=6048
    → ['Numeric tautology']
  /test?x=88 AND 2383=6355
    → ['Numeric tautology']
  /test?x=88 AND 1145=1347-- AkhR
    → ['Numeric tautology']
  /test?x=88') AND 2509=8215 AND ('NFBL'='NFBL
    → ['Numeric tautology', 'String tautology']
  /test?x=88' AND 1217=8784 AND 'zRGV'='zRGV
    → ['Boolean OR/AND', 'Numeric tautology', 'String tautology']
  /test?x=(SELECT (CASE WHEN (1429=8089) THEN 88 ELS

## 6. RF Inference — Way 3 + 3-Tier Policy

In [6]:
vectorizer = joblib.load('results/models/03_vectorizer.pkl')
rf_model   = joblib.load('results/models/03_random_forest_model.pkl')
th         = json.load(open('results/models/03_thresholds.json'))

T_HIGH = th['t_high']
T_LOW  = th['t_low']

assert T_HIGH > T_LOW, f'T_HIGH ({T_HIGH}) must be > T_LOW ({T_LOW})'
print(f'RF model   : NB03 Way3  features={rf_model.n_features_in_}')
print(f'T_high={T_HIGH}  T_low={T_LOW}  (T_high > T_low ✅)')
print()

df_qs   = df[df['query_values'].notna()].copy().reset_index(drop=True)
queries = df_qs['query_values'].tolist()

t0     = time.perf_counter()
X_text = vectorizer.transform(queries)
X_sym  = build_symbol_matrix(queries)
X_eval = hstack([X_text, X_sym])

# Align feature count
n_exp = rf_model.n_features_in_
n_act = X_eval.shape[1]
if n_act == n_exp:
    print(f'Feature check : {n_act} == {n_exp}  ✅ exact match')
elif n_act < n_exp:
    from scipy.sparse import csr_matrix as _csr
    X_eval = hstack([X_eval, _csr(np.zeros((X_eval.shape[0], n_exp - n_act)))])
    print(f'Feature check : padded {n_act} → {n_exp}  ⚠️  vectorizer/model mismatch')
else:
    X_eval = X_eval[:, :n_exp]
    print(f'Feature check : truncated {n_act} → {n_exp}  ⚠️  vectorizer/model mismatch')

scores   = rf_model.predict_proba(X_eval)[:, 1]
infer_ms = (time.perf_counter() - t0) * 1000

df_qs['rf_score'] = scores
df_qs['rf_tier']  = np.where(scores >= T_HIGH, 'ATTACK',
                    np.where(scores >= T_LOW,  'SUSPICIOUS', 'BENIGN'))

df['rf_score'] = np.nan
df['rf_tier']  = 'SKIPPED'
df.loc[df['query_values'].notna(), 'rf_score'] = df_qs['rf_score'].values
df.loc[df['query_values'].notna(), 'rf_tier']  = df_qs['rf_tier'].values

n_atk_rf  = int((df_qs['rf_tier'] == 'ATTACK').sum())
n_susp_rf = int((df_qs['rf_tier'] == 'SUSPICIOUS').sum())
n_ben_rf  = int((df_qs['rf_tier'] == 'BENIGN').sum())

print(f'Vectorize+infer : {infer_ms:.0f}ms ({infer_ms / len(df_qs) * 1000:.3f}ms per 1k entries)')
print(f'Entries scored  : {len(df_qs):,}')
print()
print(f'  ATTACK     : {n_atk_rf:,}')
print(f'  SUSPICIOUS : {n_susp_rf:,}')
print(f'  BENIGN     : {n_ben_rf:,}')
print()

if n_atk_rf + n_susp_rf > 0:
    print('RF flagged entries (ATTACK + SUSPICIOUS):')
    rf_flagged = df_qs[df_qs['rf_tier'].isin(['ATTACK', 'SUSPICIOUS'])]
    for _, row in rf_flagged.iterrows():
        lbl = f'  [true_label={row["true_label"]}]' if row['has_label'] else ''
        print(f'  score={row["rf_score"]:.3f}  {row["rf_tier"]:10s}  {row["url_decoded"][:65]}{lbl}')


RF model   : NB03 Way3  features=15202
T_high=1.0  T_low=0.99  (T_high > T_low ✅)

Feature check : 15202 == 15202  ✅ exact match
Vectorize+infer : 1093ms (121.589ms per 1k entries)
Entries scored  : 8,988

  ATTACK     : 60
  SUSPICIOUS : 36
  BENIGN     : 8,892

RF flagged entries (ATTACK + SUSPICIOUS):
  score=0.990  SUSPICIOUS  /test?x=(SELECT (CASE WHEN (1429=8089) THEN 88 ELSE (SELECT 8089 
  score=1.000  ATTACK      /test?x=88) AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b
  score=0.990  SUSPICIOUS  /test?x=88 AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b6
  score=0.990  SUSPICIOUS  /test?x=88 AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b6
  score=1.000  ATTACK      /test?x=88') AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766
  score=1.000  ATTACK      /test?x=88' AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b
  score=0.990  SUSPICIOUS  /test?x=88) AND 6509 IN (SELECT (CHAR(113)+CHAR(118)+CHAR(107)+CH
  score=0.990  SUSPICIOUS  /test?x=88 AND 6

## 7. Regex vs RF Comparison

In [7]:
# Align: re-run regex on QS-only subset for clean comparison
df_qs['regex_hits'] = df_qs['url_decoded'].apply(regex_scan)
df_qs['regex_flag'] = df_qs['regex_hits'].apply(lambda x: 1 if x else 0)
df_qs['rf_flag']    = df_qs['rf_tier'].isin(['ATTACK', 'SUSPICIOUS']).astype(int)

both_flag  = df_qs[(df_qs['regex_flag'] == 1) & (df_qs['rf_flag'] == 1)]
rf_only    = df_qs[(df_qs['regex_flag'] == 0) & (df_qs['rf_flag'] == 1)]
regex_only = df_qs[(df_qs['regex_flag'] == 1) & (df_qs['rf_flag'] == 0)]
neither    = df_qs[(df_qs['regex_flag'] == 0) & (df_qs['rf_flag'] == 0)]

total_qs = len(df_qs)
agree    = len(both_flag) + len(neither)

print('=== REGEX vs RF COMPARISON (query-string entries) ===')
print()
print(f'  Both flag  (regex ✅ + RF ✅) : {len(both_flag):,}  ← highest confidence')
print(f'  RF only    (regex ❌ + RF ✅) : {len(rf_only):,}  ← obfuscated / encoded')
print(f'  Regex only (regex ✅ + RF ❌) : {len(regex_only):,}  ← below RF threshold')
print(f'  Neither                       : {len(neither):,}  ← benign')
print()
print(f'Agreement rate: {agree / total_qs * 100:.2f}%')
print()

if len(both_flag) > 0:
    print('BOTH FLAGGED (first 15):')
    for _, row in both_flag.head(15).iterrows():
        lbl = f'  [label={row["true_label"]}]' if row['has_label'] else ''
        print(f'  score={row["rf_score"]:.3f}  {row["rf_tier"]:10s}  {row["url_decoded"][:65]}{lbl}')
        print(f'    regex: {row["regex_hits"]}')
    print()

if len(rf_only) > 0:
    print('RF ONLY — regex missed (first 10):')
    for _, row in rf_only.head(10).iterrows():
        lbl = f'  [label={row["true_label"]}]' if row['has_label'] else ''
        print(f'  score={row["rf_score"]:.3f}  {row["rf_tier"]:10s}  {row["url_decoded"][:65]}{lbl}')
    print()

if len(regex_only) > 0:
    print('REGEX ONLY — RF scored below threshold (first 10):')
    for _, row in regex_only.head(10).iterrows():
        lbl = f'  [label={row["true_label"]}]' if row['has_label'] else ''
        print(f'  score={row["rf_score"]:.3f}  {row["rf_tier"]:10s}  {row["url_decoded"][:65]}{lbl}')
        print(f'    regex: {row["regex_hits"]}')


=== REGEX vs RF COMPARISON (query-string entries) ===

  Both flag  (regex ✅ + RF ✅) : 86  ← highest confidence
  RF only    (regex ❌ + RF ✅) : 10  ← obfuscated / encoded
  Regex only (regex ✅ + RF ❌) : 86  ← below RF threshold
  Neither                       : 8,806  ← benign

Agreement rate: 98.93%

BOTH FLAGGED (first 15):
  score=0.990  SUSPICIOUS  /test?x=(SELECT (CASE WHEN (1429=8089) THEN 88 ELSE (SELECT 8089 
    regex: ['UNION SELECT']
  score=1.000  ATTACK      /test?x=88) AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b
    regex: ['SQL kw + clause', 'Subquery SELECT', 'Hex encoding', 'CONCAT', 'System tables']
  score=0.990  SUSPICIOUS  /test?x=88 AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b6
    regex: ['SQL kw + clause', 'Subquery SELECT', 'Hex encoding', 'CONCAT', 'System tables']
  score=0.990  SUSPICIOUS  /test?x=88 AND (SELECT 8421 FROM(SELECT COUNT(*),CONCAT(0x71766b6
    regex: ['SQL kw + clause', 'Subquery SELECT', 'Hex encoding', 'CONCAT', 'System ta

## 8. Full-Log Summary & Ground Truth

In [8]:
n_regex_total = int(df['regex_flag'].sum())
n_rf_atk      = int((df['rf_tier'] == 'ATTACK').sum())
n_rf_susp     = int((df['rf_tier'] == 'SUSPICIOUS').sum())
n_rf_skip     = int((df['rf_tier'] == 'SKIPPED').sum())

print('=== FULL LOG SUMMARY ===')
print()
print(f'Total entries          : {len(df):,}')
print(f'With query string      : {len(df_qs):,}')
print(f'Path-only (skipped)    : {n_rf_skip:,}')
print()
print(f'REGEX flagged          : {n_regex_total:,}  ({n_regex_total / len(df) * 100:.4f}%)')
print(f'RF ATTACK tier         : {n_rf_atk:,}')
print(f'RF SUSPICIOUS tier     : {n_rf_susp:,}')
print(f'RF Combined            : {n_rf_atk + n_rf_susp:,}')

if HAS_LABELS:
    n_true  = int((df['true_label'] == 1).sum())
    benign  = int((df['true_label'] == 0).sum())
    r_tp    = int(((df['regex_flag'] == 1) & (df['true_label'] == 1)).sum())
    r_fp    = int(((df['regex_flag'] == 1) & (df['true_label'] == 0)).sum())
    rf_tp_a = int(((df['rf_tier'] == 'ATTACK')     & (df['true_label'] == 1)).sum())
    rf_tp_s = int(((df['rf_tier'] == 'SUSPICIOUS') & (df['true_label'] == 1)).sum())
    rf_fp_a = int(((df['rf_tier'] == 'ATTACK')     & (df['true_label'] == 0)).sum())
    rf_fp_s = int(((df['rf_tier'] == 'SUSPICIOUS') & (df['true_label'] == 0)).sum())
    c_tp = rf_tp_a + rf_tp_s
    c_fp = rf_fp_a + rf_fp_s

    print()
    print('=== GROUND-TRUTH EVALUATION ===')
    print(f'True attacks           : {n_true}')
    print()
    print(f'REGEX:')
    print(f'  TP={r_tp}  FP={r_fp:,}  '
          f'Recall={r_tp / n_true:.4f}  '
          f'FP/10k={r_fp / benign * 10000:.2f}')
    print()
    print(f'RF ATTACK tier:')
    print(f'  TP={rf_tp_a}  FP={rf_fp_a:,}  '
          f'Recall={rf_tp_a / n_true:.4f}  '
          f'FP/10k={rf_fp_a / benign * 10000:.2f}')
    print(f'RF Combined (ATTACK+SUSPICIOUS):')
    print(f'  TP={c_tp}  FP={c_fp:,}  '
          f'Recall={c_tp / n_true:.4f}  '
          f'FP/10k={c_fp / benign * 10000:.2f}')


=== FULL LOG SUMMARY ===

Total entries          : 929,535
With query string      : 8,988
Path-only (skipped)    : 920,547

REGEX flagged          : 199  (0.0214%)
RF ATTACK tier         : 60
RF SUSPICIOUS tier     : 36
RF Combined            : 96


## 9. Detection Dashboard

In [9]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1. Detection counts
ax1 = fig.add_subplot(gs[0, 0])
bar_labels = ['Regex\n(full URL)', 'RF ATTACK\n(QS only)', 'RF SUSP\n(QS only)', 'RF Combined\n(QS only)']
bar_values = [n_regex_total, n_rf_atk, n_rf_susp, n_rf_atk + n_rf_susp]
bars1 = ax1.bar(bar_labels, bar_values, color=['steelblue','tomato','orange','purple'])
ax1.set_title('Detection Counts by Method')
ax1.set_ylabel('Flagged Entries')
ax1.grid(axis='y', alpha=0.4)
for b, v in zip(bars1, bar_values):
    ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.1,
             str(v), ha='center', fontsize=9, fontweight='bold')

# 2. Agreement breakdown
ax2 = fig.add_subplot(gs[0, 1])
ag_labels = ['Both flag', 'RF only', 'Regex only', 'Neither']
ag_values = [len(both_flag), len(rf_only), len(regex_only), len(neither)]
ag_colors = ['red', 'tomato', 'steelblue', 'lightgrey']
bars2 = ax2.bar(ag_labels, ag_values, color=ag_colors)
ax2.set_title('Method Agreement\n(QS entries)')
ax2.set_ylabel('Count')
ax2.grid(axis='y', alpha=0.4)
ax2.tick_params(axis='x', rotation=15)
for b, v in zip(bars2, ag_values):
    ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.1,
             str(v), ha='center', fontsize=9, fontweight='bold')

# 3. RF score distribution
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(df_qs['rf_score'], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax3.axvline(T_HIGH, color='red',    linestyle='--', linewidth=1.5, label=f'T_high={T_HIGH}')
ax3.axvline(T_LOW,  color='orange', linestyle='--', linewidth=1.5, label=f'T_low={T_LOW}')
ax3.set_xlabel('RF Score')
ax3.set_ylabel('Count')
ax3.set_title('RF Score Distribution\n(QS entries)')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# 4. 3-tier pie
ax4 = fig.add_subplot(gs[1, 0])
pie_data = [('ATTACK', n_rf_atk, 'tomato'),
            ('SUSPICIOUS', n_rf_susp, 'orange'),
            ('BENIGN', int((df['rf_tier'] == 'BENIGN').sum()), 'steelblue'),
            ('SKIPPED\n(no QS)', n_rf_skip, 'lightgrey')]
pie_data = [(l, v, c) for l, v, c in pie_data if v > 0]
ax4.pie([x[1] for x in pie_data],
        labels=[x[0] for x in pie_data],
        colors=[x[2] for x in pie_data],
        autopct='%1.2f%%', startangle=90)
ax4.set_title('3-Tier Allocation\n(full log)')

# 5. Top regex patterns
ax5 = fig.add_subplot(gs[1, 1:])
if pattern_counts:
    top = dict(pattern_counts.most_common(8))
    bars5 = ax5.barh(list(top.keys()), list(top.values()),
                     color='steelblue', alpha=0.8)
    ax5.set_xlabel('Matches')
    ax5.set_title('Top Regex Patterns Triggered')
    ax5.grid(axis='x', alpha=0.4)
    for b, v in zip(bars5, top.values()):
        ax5.text(v + 0.1, b.get_y() + b.get_height() / 2,
                 str(v), va='center', fontsize=9)
else:
    ax5.text(0.5, 0.5, 'No regex matches', ha='center', va='center',
             transform=ax5.transAxes, fontsize=12)
    ax5.set_title('Top Regex Patterns Triggered')

plt.suptitle('NB06 — Real Log: Regex vs RF Detection', fontsize=14, y=1.01)
plt.savefig('results/nb06/06_detection_dashboard.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: results/nb06/06_detection_dashboard.png')


Saved: results/nb06/06_detection_dashboard.png


## 10. Export Flagged Entries

In [10]:
flagged = df[
    (df['regex_flag'] == 1) |
    (df['rf_tier'].isin(['ATTACK', 'SUSPICIOUS']))
].copy()

flagged['rf_score'] = flagged['rf_score'].fillna(-1)

def source_label(row):
    r = int(row['regex_flag'] == 1)
    m = int(row['rf_tier'] in ('ATTACK', 'SUSPICIOUS'))
    if r and m: return 'both'
    if m:       return 'rf_only'
    if r:       return 'regex_only'
    return 'neither'

flagged['detection_source'] = flagged.apply(source_label, axis=1)

out_cols = ['url_decoded', 'query_values', 'regex_flag', 'regex_hits',
            'rf_score', 'rf_tier', 'detection_source']
if HAS_LABELS:
    out_cols = ['true_label'] + out_cols

flagged[out_cols].to_csv('results/nb06/06_flagged_entries.csv', index=False)

print(f'Flagged entries exported : {len(flagged):,}')
print(f'  both       : {(flagged["detection_source"] == "both").sum():,}')
print(f'  rf_only    : {(flagged["detection_source"] == "rf_only").sum():,}')
print(f'  regex_only : {(flagged["detection_source"] == "regex_only").sum():,}')
print()
print('Saved: results/nb06/06_flagged_entries.csv')


Flagged entries exported : 209
  both       : 86
  rf_only    : 10
  regex_only : 113

Saved: results/nb06/06_flagged_entries.csv


## 11. Summary

In [11]:
print('=' * 65)
print('NOTEBOOK 06 — COMPLETE')
print('=' * 65)
print()
print(f'Log               : {len(df):,} entries')
print(f'With query string : {len(df_qs):,}  ({len(df_qs) / len(df) * 100:.1f}%)')
print()
print(f'REGEX flagged     : {n_regex_total:,}  ({n_regex_total / len(df) * 100:.4f}%)')
print(f'RF ATTACK         : {n_rf_atk:,}')
print(f'RF SUSPICIOUS     : {n_rf_susp:,}')
print(f'RF Combined       : {n_rf_atk + n_rf_susp:,}')
print()
print(f'Agreement (QS entries):')
print(f'  Both flag  : {len(both_flag):,}  ← confirmed detections')
print(f'  RF only    : {len(rf_only):,}  ← obfuscated payloads regex missed')
print(f'  Regex only : {len(regex_only):,}  ← below RF threshold')
print(f'  Neither    : {len(neither):,}  ← benign')
print()
if HAS_LABELS:
    print('GROUND TRUTH:')
    print(f'  Regex  : recall={r_tp}/{n_true}={r_tp/n_true:.4f}  FP/10k={r_fp/benign*10000:.2f}')
    print(f'  RF atk : recall={rf_tp_a}/{n_true}={rf_tp_a/n_true:.4f}  FP/10k={rf_fp_a/benign*10000:.2f}')
    print(f'  RF comb: recall={c_tp}/{n_true}={c_tp/n_true:.4f}  FP/10k={c_fp/benign*10000:.2f}')
    print()
print('Outputs:')
print('  results/nb06/06_detection_dashboard.png')
print('  results/nb06/06_flagged_entries.csv')


NOTEBOOK 06 — COMPLETE

Log               : 929,535 entries
With query string : 8,988  (1.0%)

REGEX flagged     : 199  (0.0214%)
RF ATTACK         : 60
RF SUSPICIOUS     : 36
RF Combined       : 96

Agreement (QS entries):
  Both flag  : 86  ← confirmed detections
  RF only    : 10  ← obfuscated payloads regex missed
  Regex only : 86  ← below RF threshold
  Neither    : 8,806  ← benign

Outputs:
  results/nb06/06_detection_dashboard.png
  results/nb06/06_flagged_entries.csv
